In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Important Note**

* **Training Pipeline Cell-0 to Cell-8**
* **Inference Pipeline Cell-8.**
* **Post processing Cell-9**
* **Final submission CSV file Cell-10**

# **Cell-0 : Environment Setup : Install Dependencies**. 

In [ ]:
!pip -q install -U
!pip install -U bitsandbytes accelerate transformers
!pip install -q bitsandbytes peft accelerate evaluate jiwer

In [ ]:
import os, warnings

# Core Python

import os
import re
import glob
import gc
import unicodedata
from pathlib import Path
from dataclasses import dataclass
from typing import Any
from difflib import SequenceMatcher


# PyTorch & Audio

import torch
import torchaudio


# Data & Utilities

import pandas as pd
from tqdm.auto import tqdm
import IPython.display as ipd


# HuggingFace / Transformers

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from datasets import Dataset
import evaluate


# PEFT (LoRA)

from peft import LoraConfig, get_peft_model, PeftModel

warnings.filterwarnings("ignore")

# **Cell-1 : Project Configuration: Kaggle Paths, Inputs, and Output Folders**

In [ ]:
DATASET = Path("/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition")
DATA_ROOT = DATASET / "transcription" / "transcription"

TRAIN_AUDIO_DIR = DATA_ROOT / "train" / "audio"
TRAIN_TEXT_DIR  = DATA_ROOT / "train" / "annotation"
TEST_AUDIO_DIR  = DATA_ROOT / "test" / "audio"
SAMPLE_SUBMISSION = DATASET / "sample_submission.csv"

WORK_DIR = Path("/kaggle/working")
PROCESSED_DIR = WORK_DIR / "processed"
MODELS_DIR = WORK_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_AUDIO_DIR:", TRAIN_AUDIO_DIR)
print("TRAIN_TEXT_DIR :", TRAIN_TEXT_DIR)
print("TEST_AUDIO_DIR :", TEST_AUDIO_DIR)
print("SAMPLE_SUBMISSION:", SAMPLE_SUBMISSION)

print("Train wav:", len(list(TRAIN_AUDIO_DIR.glob("train_*.wav"))))
print("Train txt:", len(list(TRAIN_TEXT_DIR.glob("train_*.txt"))))
print("Test  wav:", len(list(TEST_AUDIO_DIR.glob("test_*.wav"))))


# **Cell-2 : Dataset Preparation: Split Long Train Audio into Fixed 30-Second Chunks**

> **Workflow**
1. Converts audio to mono 16kHz
2. Splits long training audio into fixed 30-second segments
3. Skips very short trailing fragments
4. Saves chunked WAV files
5. Generates structured CSV metadata
6. Prepares dataset for model training

In [ ]:
OUT_AUDIO_DIR = PROCESSED_DIR / "train_fixed30s_audio"
OUT_CSV = PROCESSED_DIR / "train_fixed30s.csv"
OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

NUM_FILES = 112        
TARGET_SR = 16000
CHUNK_SEC = 30.0
MIN_SEC   = 1.0

def read_text_safe(path):
    path = str(path)
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return re.sub(r"\s+", " ", f.read()).strip()

def load_audio_mono_16k(path, target_sr=16000):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.squeeze(0)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
        sr = target_sr
    return wav, sr

wav_files = sorted(glob.glob(str(TRAIN_AUDIO_DIR / "train_*.wav")))[:NUM_FILES]
print("Found wavs:", len(wav_files))

all_rows = []
for ONE_WAV in tqdm(wav_files, desc="Fixed 30s chunking"):
    file_id = os.path.splitext(os.path.basename(ONE_WAV))[0]
    ONE_TXT = str(TRAIN_TEXT_DIR / f"{file_id}.txt")

    file_out_dir = OUT_AUDIO_DIR / file_id
    file_out_dir.mkdir(parents=True, exist_ok=True)

    text = read_text_safe(ONE_TXT)

    wav, sr = load_audio_mono_16k(ONE_WAV, TARGET_SR)
    total_samples = len(wav)

    chunk_samples = int(CHUNK_SEC * sr)
    min_samples   = int(MIN_SEC * sr)

    chunk_idx = 0
    for start in range(0, total_samples, chunk_samples):
        end = min(start + chunk_samples, total_samples)
        if (end - start) < min_samples:
            continue

        chunk = wav[start:end]

        s_ms = int((start / sr) * 1000)
        e_ms = int((end / sr) * 1000)

        out_path = file_out_dir / f"{file_id}_30s_{chunk_idx:05d}_{s_ms}ms_{e_ms}ms.wav"
        torchaudio.save(str(out_path), chunk.unsqueeze(0), sr)

        all_rows.append({
            "audio": str(out_path),
            "text": text,
            "start_sec": start / sr,
            "end_sec": end / sr,
            "duration_sec": (end - start) / sr
        })
        chunk_idx += 1

df_chunks = pd.DataFrame(all_rows)
df_chunks.to_csv(OUT_CSV, index=False)
print("Saved chunk audios to:", OUT_AUDIO_DIR)
print("Saved CSV to:", OUT_CSV)
df_chunks.head()

# **Cell-3 : Speech Extraction + Baseline ASR: VAD Merge and Whisper Transcription on Chunks**

> **Workflow**
1. Loads fine-tuned Whisper model
2. Integrates Silero Voice Activity Detection (VAD)
3. Removes silence before transcription
4. Transcribes only detected speech regions
5. Generates chunk-level transcripts
6. Saves structured CSV for training

In [ ]:
INPUT_CSV = PROCESSED_DIR / "train_fixed30s.csv"
OUT_CSV = PROCESSED_DIR / "train_fixed30s_vad_transcribed.csv"

MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Loading Whisper...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()

print("Loading Silero VAD...")
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    onnx=False
)
(get_speech_timestamps, _, _, _, collect_chunks) = utils
vad_model = vad_model.to(device).eval()

def load_and_resample(path, target_sr=16000):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.squeeze(0)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav

def process_chunk(path):
    wav = load_and_resample(path)

    with torch.inference_mode():
        wav_gpu = wav.to(device)
        speech_timestamps = get_speech_timestamps(
            wav_gpu, vad_model, sampling_rate=16000, threshold=0.5
        )

    if not speech_timestamps:
        return ""

    merged_wav = collect_chunks(speech_timestamps, wav_gpu)

    input_features = processor(
        merged_wav.cpu().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device, dtype=torch.float16 if device == "cuda" else torch.float32)

    with torch.inference_mode():
        pred_ids = model.generate(input_features, max_new_tokens=256)

    return processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

df = pd.read_csv(INPUT_CSV)
print("Processing chunks:", len(df))

transcripts = []
for audio_path in tqdm(df["audio"].tolist(), desc="VAD+Transcribe"):
    try:
        transcripts.append(process_chunk(audio_path))
    except Exception as e:
        print("Error:", audio_path, e)
        transcripts.append("")

df["transcript"] = transcripts
final_df = df[["audio", "start_sec", "end_sec", "duration_sec", "transcript"]].rename(columns={"audio": "filename"})
final_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
final_df.head()

# **Cell-4 : Ground-Truth Alignment: Map Chunk ASR Back to Original Train Annotations**

> **Workflow**
1. Bengali text normalization
2. Word-level tokenization
3. Sliding-window search over original transcript
4. Fuzzy similarity scoring using SequenceMatcher
5. Context-aware cursor progression monotonic alignment
6. Replacing noisy ASR output with best-matched ground-truth snippet
7. Storing alignment confidence score

In [ ]:
ALIGNED_CSV = PROCESSED_DIR / "train_fixed30s_vad_transcribed.csv"
OUT_CSV = PROCESSED_DIR / "train_fixed30s_aligned_final.csv"

def norm_bn(s: str) -> str:
    s = str(s or "").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"[^\u0980-\u09FF0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def to_words(s: str):
    return norm_bn(s).split()

SEARCH_AHEAD_WORDS = 800
MIN_QUERY_WORDS = 3
LEN_TOL = 0
PAD_WORDS = 1
SCORE_OK = 0.50

def best_match_fixed_len(orig_words, query_words, start_w, search_ahead_words, len_tol=0):
    segment = orig_words[start_w : min(len(orig_words), start_w + search_ahead_words)]
    if not segment:
        return None, None, 0.0

    qlen = len(query_words)
    sizes = [max(3, qlen + d) for d in range(-len_tol, len_tol + 1)]

    best = (None, None, 0.0)
    stride = 1 if qlen <= 12 else 2 if qlen <= 30 else 3

    for i in range(0, max(1, len(segment) - min(sizes)), stride):
        for wlen in sizes:
            j = i + wlen
            if j > len(segment):
                continue
            cand = segment[i:j]
            score = SequenceMatcher(None, cand, query_words).ratio()
            if score > best[2]:
                best = (start_w + i, start_w + j, score)
    return best

def get_train_id(row):
    fname = os.path.basename(str(row["filename"]))
    if "_30s_" in fname:
        return fname.split("_30s_")[0]
    parts = fname.split("_")
    return parts[0] + "_" + parts[1] if len(parts) >= 2 else fname

df = pd.read_csv(ALIGNED_CSV)
df["train_id"] = df.apply(get_train_id, axis=1)

print("Loaded chunks:", len(df), "| Unique train files:", df["train_id"].nunique())

orig_cache = {}
replaced_all = [""] * len(df)
scores_all = [0.0] * len(df)

for train_id, g in tqdm(df.groupby("train_id", sort=False), desc="Aligning Text"):
    txt_path = str(TRAIN_TEXT_DIR / f"{train_id}.txt")
    if not os.path.exists(txt_path):
        continue

    if train_id not in orig_cache:
        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            orig_cache[train_id] = to_words(f.read())

    orig_words = orig_cache[train_id]
    cursor = 0
    g = g.sort_values("start_sec") if "start_sec" in g.columns else g.sort_index()

    for idx in g.index:
        q_words = to_words(df.loc[idx, "transcript"])

        if len(q_words) < MIN_QUERY_WORDS:
            replaced_all[idx] = ""
            scores_all[idx] = 0.0
            continue

        b0, b1, sc = best_match_fixed_len(orig_words, q_words, cursor, SEARCH_AHEAD_WORDS, LEN_TOL)
        if sc < SCORE_OK:
            b0b, b1b, scb = best_match_fixed_len(orig_words, q_words, cursor, SEARCH_AHEAD_WORDS * 3, LEN_TOL)
            if scb > sc:
                b0, b1, sc = b0b, b1b, scb

        if b0 is None:
            continue

        s = max(0, b0 - PAD_WORDS)
        e = min(len(orig_words), b1 + PAD_WORDS)
        snippet = " ".join(orig_words[s:e]).strip()

        if sc < SCORE_OK:
            snippet = " ".join(orig_words[b0:b1]).strip()

        replaced_all[idx] = snippet
        scores_all[idx] = sc

        if sc >= SCORE_OK:
            cursor = max(cursor, b1)

df["orig_replaced_text"] = replaced_all
df["match_score"] = scores_all

df.to_csv(OUT_CSV, index=False)
print("Saved aligned dataset:", OUT_CSV)
df[["train_id", "transcript", "orig_replaced_text", "match_score"]].head()


# **Cell-5 : Training Data Builder: Create Clean train_final.csv for Fine-Tuning**

> **Workflow**
1. Selecting only required columns (filename, aligned transcript)
2. Renaming to standardized training format
3. Removing empty or NaN transcripts
4. Producing the final clean training CSV

In [ ]:
INPUT_CSV = PROCESSED_DIR / "train_fixed30s_aligned_final.csv"
FINAL_CSV = PROCESSED_DIR / "train_final.csv"

df = pd.read_csv(INPUT_CSV)

final_df = df[["filename", "orig_replaced_text"]].rename(
    columns={"orig_replaced_text": "transcript"}
)

print("Rows before cleaning:", len(final_df))
final_df = final_df.dropna(subset=["transcript"])
final_df = final_df[final_df["transcript"].astype(str).str.strip() != ""]
print("Rows after cleaning :", len(final_df))

final_df.to_csv(FINAL_CSV, index=False)
print("Saved:", FINAL_CSV)
final_df.head()


# **Cell-6 : Sanity Check: Listen to Random Samples and Verify Audio Paths**

> **Workflow**
1. Loads the final training CSV
2. Checks for missing audio files
3. Randomly samples a few entries
4. Displays transcript preview
5. Plays audio for manual verification

In [ ]:
CSV_PATH = PROCESSED_DIR / "train_final.csv"
df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

missing = [f for f in df["filename"] if not os.path.exists(f)]
print("Missing files:", len(missing))

n = min(3, len(df))
samples = df.sample(n=n, random_state=42)

for idx, row in samples.iterrows():
    fname = row["filename"]
    text = row["transcript"]
    print("\n--- Sample", idx, "---")
    print("File:", os.path.basename(fname))
    print("Text:", text[:200], "..." if len(text) > 200 else "")
    ipd.display(ipd.Audio(fname))


# **Cell-7 : Fine-Tuning: Train Whisper-Medium with LoRA (Adapter Training)**

> **Workflow**
1. GPU memory cleanup
2. Train–eval split (90/10)
3. Whisper model loading
4. LoRA adapter injection (PEFT)
5. Audio feature extraction & tokenization
6. Custom speech data collator
7. WER evaluation metric
8. Seq2Seq training configuration
9. Training with gradient checkpointing + fp16
10. Saving LoRA adapter & processor

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
CHUNK_CSV = str(PROCESSED_DIR / "train_final.csv")
OUTPUT_DIR = str(MODELS_DIR / "whisper-bengali-lora")
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if device == "cuda" else torch.float32
MAX_LABEL_LEN = 448

df = pd.read_csv(CHUNK_CSV).sample(frac=1, random_state=42).reset_index(drop=True)
train_limit = int(len(df) * 0.9)
train_df, eval_df = df.iloc[:train_limit], df.iloc[train_limit:]

train_dataset = Dataset.from_pandas(train_df)
eval_dataset  = Dataset.from_pandas(eval_df)

print("Train:", len(train_dataset), "| Eval:", len(eval_dataset))

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
).to(device)
model.config.use_cache = False

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def prepare_dataset(batch):
    try:
        audio_path = batch["filename"]
        wav, sr = torchaudio.load(audio_path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)

        batch["input_features"] = processor.feature_extractor(
            wav.numpy(), sampling_rate=16000
        ).input_features[0]

        batch["labels"] = processor.tokenizer(
            batch["transcript"], truncation=True, max_length=MAX_LABEL_LEN
        ).input_ids
    except Exception as e:
        batch["input_features"] = None
        batch["labels"] = None
    return batch

train_dataset = train_dataset.map(prepare_dataset)
eval_dataset  = eval_dataset.map(prepare_dataset)

train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)
eval_dataset  = eval_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    feature_dtype: torch.dtype = torch.float16

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].to(self.feature_dtype)

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, feature_dtype=MODEL_DTYPE)

wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=3000,
    gradient_checkpointing=True,
    fp16=(device == "cuda"),
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=25,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Starting LoRA training...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter + processor to:", OUTPUT_DIR)


# **Cell-8 : Inference Pipeline: LoRA + VAD + Sliding-Window Transcription on Test Audio**

> **Workflow**
1. Loading base Whisper + trained LoRA adapter
2. Setting Bengali forced decoding configuration
3. Integrating Silero VAD for silence removal
4. Sliding window chunking with overlap (20s, 1s stride overlap)
5. Batch inference with mixed precision
6. Beam search decoding (num_beams=5)
7. Reconstructing full transcript per audio file
8. Generating final competition submission CSV

In [ ]:
BASE_MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
ADAPTER_DIR = str(MODELS_DIR / "whisper-bengali-lora")  # from Cell 7
OUT_CSV = str(PROCESSED_DIR / "submission_raw.csv")

BATCH_SIZE = 16
CHUNK_SEC = 20.0
OVERLAP_SEC = 1.0
SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if device == "cuda" else torch.float32
print("Device:", device, "| Batch:", BATCH_SIZE)

print("Loading base + LoRA...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, torch_dtype=MODEL_DTYPE, low_cpu_mem_usage=True
).to(device)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="bengali", task="transcribe")
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.suppress_tokens = []
model.generation_config.begin_suppress_tokens = []
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None

print("Loading VAD...")
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    onnx=False,
)
(get_speech_timestamps, _, _, _, collect_chunks) = utils
vad_model = vad_model.to(device).eval()

print("Preparing chunk metadata...")
test_files = sorted(glob.glob(str(TEST_AUDIO_DIR / "*.wav")))
all_chunks = []

for wav_path in tqdm(test_files, desc="Metadata"):
    file_id = os.path.splitext(os.path.basename(wav_path))[0]
    info = torchaudio.info(wav_path)
    total_samples = info.num_frames
    orig_sr = info.sample_rate

    target_samples = int(total_samples * (SR / orig_sr))
    chunk_samples = int(CHUNK_SEC * SR)
    stride_samples = int((CHUNK_SEC - OVERLAP_SEC) * SR)

    for start in range(0, target_samples, stride_samples):
        end = min(start + chunk_samples, target_samples)
        if (end - start) < int(1.0 * SR):
            continue
        all_chunks.append({"file_id": file_id, "path": wav_path, "start": start, "end": end})

print("Total chunks:", len(all_chunks))

def load_and_crop(path, start, end):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.squeeze(0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    return wav[start:end]

def apply_vad_and_merge(wav_tensor):
    if wav_tensor.numel() == 0:
        return None
    with torch.inference_mode():
        wav_gpu = wav_tensor.to(device)
        timestamps = get_speech_timestamps(wav_gpu, vad_model, sampling_rate=SR, threshold=0.5)
        if not timestamps:
            return None
        merged = collect_chunks(timestamps, wav_gpu)
        return merged.cpu()

results_map = {fid: [] for fid in set(c["file_id"] for c in all_chunks)}

for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Batch Inference"):
    batch_meta = all_chunks[i:i + BATCH_SIZE]
    valid_features, valid_indices = [], []

    for idx, meta in enumerate(batch_meta):
        raw_wav = load_and_crop(meta["path"], meta["start"], meta["end"])
        clean_wav = apply_vad_and_merge(raw_wav)
        if clean_wav is None or clean_wav.numel() == 0:
            continue

        feat = processor(
            clean_wav.numpy(),
            sampling_rate=SR,
            return_tensors="pt",
        ).input_features[0]
        valid_features.append(feat)
        valid_indices.append(idx)

    if not valid_features:
        continue

    input_tensor = torch.stack(valid_features).to(device, dtype=MODEL_DTYPE)

    with torch.inference_mode():
        if device == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                generated_ids = model.generate(
                    input_tensor, max_new_tokens=128, num_beams=5, do_sample=False
                )
        else:
            generated_ids = model.generate(
                input_tensor, max_new_tokens=128, num_beams=5, do_sample=False
            )

    transcripts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for j, text in enumerate(transcripts):
        meta_idx = valid_indices[j]
        file_id = batch_meta[meta_idx]["file_id"]
        results_map[file_id].append(text.strip())

    del input_tensor, generated_ids, valid_features
    if device == "cuda" and (i % 10 == 0):
        torch.cuda.empty_cache()

final_rows = []
for file_id in sorted(results_map.keys()):
    full_text = re.sub(r"\s+", " ", " ".join(results_map[file_id])).strip()
    final_rows.append({"filename": file_id, "transcript": full_text})

df_sub = pd.DataFrame(final_rows)
df_sub.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("Saved raw submission:", OUT_CSV)
df_sub.head()

# **Cell-9 : Post-Processing: Clean Repetitions, Fix Common Errors, Normalize Text**

> **Workflow**
1. Unicode normalization & corrupted character cleanup
2. Phrase level corrections common ASR misrecognitions.
3. Word level correction mapping
4. Filler-word handling
5. Repeated token suppression (run capping)
6. N-gram loop removal repetition collapse
7. Overlap echo removal chunk boundary duplication fix
8. Adaptive cleaning strength based on text diversity
9. Final cleaned submission CSV generation

In [ ]:
INPUT_CSV = str(PROCESSED_DIR / "submission_raw.csv")
OUTPUT_CSV = str(PROCESSED_DIR / "submission_clean.csv")

def norm_key(x: str) -> str:
    x = unicodedata.normalize("NFC", str(x)).lower()
    x = re.sub(r"[^\u0980-\u09FFa-z0-9]", "", x)
    return x

FILLERS = {"ভাই","না","হ্যাঁ","এই","ওই","আরে","মানে","ওকে","প্লিজ","চলো","দাঁড়া","দাঁড়া","হুম"}
FILLER_KEYS = {norm_key(x) for x in FILLERS}

PHRASE_FIXES = [
    (r"এক্সকিউরি", "এক্সকিউজ মি"),
    (r"ড্রেস টুস", "ড্রেস"),
    (r"সেলস ম্যান", "সেলসম্যান"),
    (r"গুড মর্ডিং", "গুড মর্নিং"),
    (r"হেরেজমেন্ট|হেরেজমেন|হেরেজমেনট", "হ্যারাসমেন্ট"),
]

WORD_FIXES = {
    "রেসিভ": "রিসিভ",
    "পাসে": "পাশে",
    "সেন্টার": "সেন্টার",
    "বিজনেসস": "বিজনেস",
    "স্যরি": "সরি",
}

def normalize_text(t: str) -> str:
    t = unicodedata.normalize("NFC", str(t or ""))
    t = t.replace("\uFFFD", "")
    t = re.sub(r"<[^>]*>", " ", t)
    t = t.replace("<>", " ")
    t = re.sub(r"[\x00-\x1F\x7F]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def apply_maps(text: str) -> str:
    for p, r in PHRASE_FIXES:
        text = re.sub(p, r, text)
    return text

def cap_runs(tokens, short_max=1, normal_max=2):
    out = []
    prev, run = None, 0
    for tok in tokens:
        k = norm_key(tok)
        if not k:
            continue
        if k in FILLER_KEYS:
            out.append(tok)
            prev, run = None, 0
            continue
        if k == prev:
            run += 1
        else:
            prev, run = k, 1
        mx = short_max if len(k) <= 2 else normal_max
        if run <= mx:
            out.append(tok)
    return out

def remove_ngram_loops(tokens, min_n=2, max_n=12, min_repeats=2):
    norms = [norm_key(t) for t in tokens]
    out, i, L = [], 0, len(tokens)
    while i < L:
        reduced = False
        for n in range(min(max_n, L - i), min_n - 1, -1):
            if i + 2 * n > L:
                continue
            base = norms[i:i+n]
            if not any(base):
                continue
            if any(x in FILLER_KEYS for x in base):
                continue
            reps, j = 1, i + n
            while j + n <= L and norms[j:j+n] == base:
                reps += 1
                j += n
            if reps >= min_repeats:
                out.extend(tokens[i:i+n])
                i = j
                reduced = True
                break
        if not reduced:
            out.append(tokens[i])
            i += 1
    return out

def remove_overlap_echo(tokens, min_w=4, max_w=16, sim_thr=0.94):
    norms = [norm_key(t) for t in tokens]
    out_t, out_n = [], []
    i, L = 0, len(tokens)
    while i < L:
        skipped = False
        wmax = min(max_w, len(out_n), L - i)
        for w in range(wmax, min_w - 1, -1):
            tail = out_n[-w:]
            head = norms[i:i+w]
            if any(x in FILLER_KEYS for x in tail) or any(x in FILLER_KEYS for x in head):
                continue
            if tail == head:
                i += w
                skipped = True
                break
            if SequenceMatcher(None, " ".join(tail), " ".join(head)).ratio() >= sim_thr:
                i += w
                skipped = True
                break
        if not skipped:
            out_t.append(tokens[i])
            out_n.append(norms[i])
            i += 1
    return out_t

def postprocess(t: str) -> str:
    t = normalize_text(t)
    t = apply_maps(t)
    tokens = t.split()

    mapped = []
    for tok in tokens:
        k = norm_key(tok)
        mapped.append(WORD_FIXES.get(k, tok))
    tokens = mapped

    norms = [norm_key(x) for x in tokens if norm_key(x)]
    uniq_ratio = len(set(norms)) / max(len(norms), 1)
    aggressive = uniq_ratio < 0.45

    if aggressive:
        tokens = cap_runs(tokens, short_max=1, normal_max=1)
        tokens = remove_ngram_loops(tokens, max_n=14, min_repeats=2)
        tokens = remove_overlap_echo(tokens, max_w=20, sim_thr=0.90)
    else:
        tokens = cap_runs(tokens, short_max=1, normal_max=2)
        tokens = remove_ngram_loops(tokens, max_n=10, min_repeats=2)
        tokens = remove_overlap_echo(tokens, max_w=14, sim_thr=0.96)

    tokens = cap_runs(tokens, short_max=1, normal_max=2)
    return normalize_text(" ".join(tokens))

df = pd.read_csv(INPUT_CSV)
df["transcript"] = df["transcript"].fillna("").map(postprocess)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved cleaned submission:", OUTPUT_CSV)
print("Remaining � count:", df["transcript"].str.count("\uFFFD").sum())
df.head()


# **Cell-10 : Submission Builder: Match Sample Order and Export Final**

In [ ]:
CLEAN_SUB = PROCESSED_DIR / "submission_clean.csv"
FINAL_SUB = WORK_DIR / "submission.csv"

sample = pd.read_csv(SAMPLE_SUBMISSION)
pred = pd.read_csv(CLEAN_SUB)


final = sample[["filename"]].merge(pred, on="filename", how="left")
final["transcript"] = final["transcript"].fillna("")

final.to_csv(FINAL_SUB, index=False, encoding="utf-8-sig")
print("Final submission saved to:", FINAL_SUB)
final.head()
